<a href="https://colab.research.google.com/github/mostofa89/Pytorch_Fundamentals/blob/main/Claasification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sklearn
from sklearn.datasets import make_circles
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split

In [ ]:
n_samples = 1000

X, y = make_circles(n_samples, noise = 0.03, random_state = 42)

In [ ]:
len(X), len(y)

In [ ]:
print(f"First 5 samples of X:\n {X[:5]}")
print(f"First 5 samples of y:\n {y[:5]}")

In [ ]:
print(y)

In [ ]:
circles = pd.DataFrame({
    "X1" : X[ : , 0],
    "X2" : X[ : , 1],
    "label" : y
})

circles.head(10)

In [ ]:
plt.scatter(
    x = X[ : , 0],
    y = X[ : , 1],
    c = y,
    cmap = plt.cm.RdYlBu
)

plt.show()

In [ ]:
X.shape, y.shape

In [ ]:
x_sample = X[0]
y_sample = y[0]

print(f"Values for one sample of X : {x_sample} and same for y : {y_sample}")
print(f"Shape for one sample of X : {x_sample.shape} and same for y : {y_sample.shape}")

In [ ]:
type(X)

In [ ]:
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

X[ : 5], y[ : 5]

In [ ]:
X.dtype, y.dtype

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state=42
)

len(X_train), len(X_test), len(y_train), len(y_test)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else "cpu"

device

In [ ]:
class ClassifiactionV1(nn.Module):

  def __init__(self):
    super().__init__()
    self.layer_1 = nn.Linear(in_features=2, out_features=5)
    self.layer_2 = nn.Linear(in_features=5, out_features=1)


  def forward(self, x):
    return self.layer_2(self.layer_1(x))


model_1 = ClassifiactionV1().to(device)
model_1

In [ ]:
model_1 = nn.Sequential(
    nn.Linear(in_features=2, out_features=5),
    nn.Linear(in_features=5, out_features=1)
).to(device)

model_1

In [ ]:
model_1.state_dict()

In [ ]:
with torch.inference_mode():
  untrain_pred = model_1(X_test.to(device))

untrain_pred [ : 10]

In [ ]:
loss_func = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(), lr = 0.1)

In [ ]:
def accuracy_func(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct / len(y_pred)) * 100

  return acc

In [ ]:
model_1.eval()
with torch.inference_mode():
  y_logits = model_1(X_test.to(device))[ : 5]

y_logits

In [ ]:
y_pred_prob = torch.sigmoid(y_logits)

y_pred_prob

In [ ]:
y_pred = torch.round(y_pred_prob)
y_pred_label = torch.round(torch.sigmoid(model_1(X_test.to(device))))[ : 5]

print(torch.eq(y_pred.squeeze(), y_pred_label.squeeze()))
y_pred.squeeze()

In [ ]:
y_test[ : 5]

In [ ]:
device

In [ ]:
!nvidia-smi

In [ ]:
torch.cuda.manual_seed(42)

epochs = 100

X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(epochs):

  # Train The Model
  model_1.train()

  # Forward Props
  y_logits = model_1(X_train).squeeze()
  y_pred = torch.round(torch.sigmoid(y_logits))

  # Calculating Loss
  loss = loss_func(
      y_logits,
      y_train
  )

  acc = accuracy_func(
      y_true = y_train,
      y_pred = y_pred
  )

  # Optimizer backward
  loss.backward()

  #Optimizer step
  optimizer.step()


  model_1.eval()
  with torch.inference_mode():
    test_logits = model_1(X_test).squeeze()
    test_pred = torch.round(torch.sigmoid(test_logits))

    test_loss = loss_func(
        test_logits,
        y_test
    )

    test_acc = accuracy_func(
        y_true = y_test,
        y_pred = test_pred
    )

    if epoch % 10 == 0:
      print(f"Epoch : {epoch} | Loss : {loss:.5f} and Accuracy : {acc:.5f}% | Test Loss : {test_loss: .5f} and Test Accuracy : {test_acc: .5f}%")

In [ ]:
class CircleModelv1(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer_1 = nn.Linear(in_features=2, out_features=10)
    self.layer_2 = nn.Linear(in_features=10, out_features=10)
    self.layer_3 = nn.Linear(in_features=10, out_features=1)


  def forward(self, x):
    return self.layer_3(self.layer_2(self.layer_1(x)))


model_2 = CircleModelv1().to(device)
model_2

In [ ]:
model_2.state_dict()

In [ ]:
loss_func = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(params=model_2.parameters(), lr = 0.01)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

epochs = 1000

X_train, y_tain = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

for epoch in range(epochs):
  model_2.train()

  y_logits = model_2(X_train).squeeze()
  y_preds = torch.round(torch.sigmoid(y_logits))

  loss = loss_func(y_logits, y_train)
  acc = accuracy_func(y_true=y_train, y_pred=y_preds)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  model_2.eval()

  with torch.inference_mode():
    test_logits = model_2(X_test).squeeze()
    test_preds = torch.round(torch.sigmoid(test_logits))

    test_loss = loss_func(test_logits, test_preds)
    test_acc = accuracy_func(y_true=y_test, y_pred=test_preds)

    if epoch % 100 == 0:
      if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%")

In [ ]:
weight = 0.7
bias = 0.3
start = 0
end = 1
step = 0.01

X_regression = torch.arange(start, end, step).unsqueeze(dim=1)
y_regression = weight * X_regression + bias

X_regression[ : 5], y_regression[ : 5]

In [ ]:
train_split = int(0.8 * len(X_regression))
X_train_regression, y_train_regression = X_regression[ : train_split], y_regression[ : train_split]
X_test_regression, y_test_regression = X_regression[train_split : ], y_regression[train_split : ]

len(X_train_regression), len(y_train_regression)

In [ ]:
model_3 = nn.Sequential(
    nn.Linear(in_features=1, out_features=10),
    nn.Linear(in_features=10, out_features=10),
    nn.Linear(in_features=10, out_features=1)

).to(device)

model_3

In [ ]:
loss_func = nn.L1Loss()
optimizer = torch.optim.SGD(params=model_3.parameters(), lr = 0.01)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

X_train_regression, y_train_regression = X_train_regression.to(device), y_train_regression.to(device)
X_test_regression, y_test_regression = X_test_regression.to(device), y_test_regression.to(device)

epochs = 1000

for epoch in range(epochs):
  y_pred = model_3(X_train_regression)
  loss = loss_func(y_pred, y_train_regression)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  model_3.eval()
  with torch.inference_mode():
    test_pred = model_3(X_test_regression)
    test_loss = loss_func(test_pred, y_test_regression)

  if epoch % 100 == 0:
    print(f"Epoch : {epoch} | Loss : {loss:.5f} | Test loss : {test_loss:.5f}")